# Grocery Receipt Tracker on Colab GPU
Run the cells **top to bottom**. Connect this notebook to a Colab **T4 GPU** runtime first.

To stop the app, stop the last cell. Records are saved to your Google Drive when it's available.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# Should print something like "Tesla T4, 15360 MiB".
# If it errors, the runtime has no GPU: reconnect and pick T4 GPU.

## 2. Get the latest code from GitHub

In [ ]:
import os

REPO = "https://github.com/raman1893/grocery-receipt-tracker.git"
APP_DIR = "/content/grocery-receipt-tracker"

if os.path.isdir(APP_DIR):
    os.system(f"git -C {APP_DIR} pull")      # already downloaded: fetch your latest changes
else:
    os.system(f"git clone {REPO} {APP_DIR}")
os.chdir(APP_DIR)
print("Files:", sorted(os.listdir(APP_DIR)))

## 3. Install packages
PyTorch is already installed on Colab, so it isn't reinstalled here. Re-run this cell after pulling new code, in case requirements changed.

In [ ]:
!pip install -q -U "transformers>=4.57" gradio plotly openpyxl

## 4. Keep records in Google Drive
If Drive can't connect, the app still works, but records last only for this session. Use **Download Excel** before you close.

In [ ]:
DB_DIR = "/content"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DB_DIR = "/content/drive/MyDrive/grocery-receipt-tracker"
    os.makedirs(DB_DIR, exist_ok=True)
    print("Records will be saved to Google Drive:", DB_DIR)
except Exception as e:
    print("Google Drive not available, so records last only this session.")
    print("Reason:", e)

os.environ["GROCERY_DB"] = f"{DB_DIR}/grocery.db" 

## 5. Start the app
You'll be asked to choose a password (it isn't saved anywhere). The app starts in the background and this cell prints the link, then finishes. Open the link and log in with username `receipts`. The first start downloads the model (about 9 GB), which takes a few minutes.

In [ ]:
import re, time, subprocess
from getpass import getpass

os.environ["GRADIO_SHARE"] = "1"
os.environ["APP_PASSWORD"] = getpass("Choose a password for the app: ")

os.system("pkill -f '[a]pp[.]py'")          # stop any copy that's still running
time.sleep(3)

LOG = "/content/app.log"
app_proc = subprocess.Popen(["python", "-u", "app.py"], cwd=APP_DIR,
                            stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("Starting the app...", end="", flush=True)

url = None
for _ in range(900):                      # wait up to 15 minutes
    time.sleep(2)
    log = open(LOG).read()
    found = re.search(r"https://[\w.-]+\.gradio\.live", log)
    if found:
        url = found.group(0)
        break
    if app_proc.poll() is not None:       # the app stopped with an error
        break
    print(".", end="", flush=True)

if url:
    print(f"\n\nReady. Open this link and log in with username 'receipts':\n{url}")
else:
    print("\n\nThe app didn't start. The end of its log:\n")
    print(log[-3000:])

## 6. Stop the app
Run this when you're done, or before starting the app again. Then disconnect the Colab server so the free GPU time isn't used up.

In [ ]:
os.system("pkill -f '[a]pp[.]py'")
print("App stopped.")

## 7. If something goes wrong, see the app's log

In [ ]:
print(open("/content/app.log").read()[-5000:])